Additional evaluations on outputs, in addition to those in model training/cv

In [ ]:
import pandas as pd
import re
import os
from tensorflow.keras.layers import Dot, Activation
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
# import tensorflow_recommenders as tfrs

# 👉 everything below comes from *tf.keras*
from tensorflow import keras
from tensorflow.keras.utils   import FeatureSpace
from tensorflow.keras.layers  import TextVectorization

from keras_rs.layers import BruteForceRetrieval
from keras_rs.metrics import PrecisionAtK, RecallAtK
from keras.losses import BinaryFocalCrossentropy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model            import LogisticRegression
from sklearn.metrics                 import roc_auc_score

import numpy as np, pandas as pd, tensorflow as tf, keras
# from keras.layers import TextVectorization, Embedding, Concatenate
# from keras.utils  import FeatureSpace
from sklearn.metrics import classification_report

In [ ]:
DATA_DIR = "../data/opentargets/"

In [ ]:
# df_learn = pd.read_parquet("../data/proc/df_learn.parquet")
# print(df_learn.shape)
# display(df_learn)
disease_df = pd.read_parquet("../data/proc/disease_df.parquet")
print(disease_df.shape)
display(disease_df.head(2))
target_df = pd.read_parquet("../data/proc/target_df.parquet")
print(target_df.shape)
display(target_df.head(2))

In [ ]:
# Ensure columns are treated as strings to avoid errors with NaN values
disease_df['diseaseId'] = disease_df['diseaseId'].astype(str)
disease_df['dbXRefs'] = disease_df['dbXRefs'].astype(str)

# --- ONE-LINERS START ---

# 1. Define Rare Disease (Exact match for Orphanet ID or reference)
disease_df['is_rare'] = disease_df['diseaseId'].str.contains('Orphanet|ORPHA') | disease_df['dbXRefs'].str.contains('Orphanet|ORPHA')
print(disease_df['is_rare'].value_counts())
# 2. Define Mendelian/Simple Disease (Exact match for OMIM ID or reference)
disease_df['has_omim_annotation'] = disease_df['diseaseId'].str.contains('OMIM') | disease_df['dbXRefs'].str.contains('OMIM')
print(disease_df['has_omim_annotation'].value_counts())


In [ ]:
df_preds = pd.read_csv("DL_novel_candidates_predictions.csv") # DL preds, novel and positive
df_preds

In [ ]:
df_novel = df_preds.query("label<1")
df_novel

In [ ]:
df_known_pos = df_preds.query("label>0")
df_known_pos


Load targetability - and max clinical phase reached
- max clinical trial pahse is per target, NOT target X disease

In [ ]:
target_priority = pd.read_parquet(os.path.join(DATA_DIR, "target_prioritisation")).dropna(subset=["maxClinicalTrialPhase"])
target_priority

In [ ]:
temp = df_known_pos.merge(target_priority,on=["targetId"],how="inner")
print(temp.shape[0])
print(temp.maxClinicalTrialPhase.corr(temp.score))